# PREDICT-Play | R-GCN

Run this notebook after `02_feature_engineering.ipynb`.


## Imports and load data

In [15]:
# Import the modelling libraries.

from pathlib import Path
import copy
import time
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [17]:
# Load the shared title features and graph files created in notebook 02.
model_features = pd.read_csv( "https://raw.githubusercontent.com/bjvvuuren/predict-play/refs/heads/main/modelling/datasets/feature-engineering/model_features.csv")
rgcn_node_features = pd.read_csv("https://raw.githubusercontent.com/bjvvuuren/predict-play/refs/heads/main/modelling/datasets/feature-engineering/rgcn_node_features.csv")
rgcn_edges = pd.read_csv("https://raw.githubusercontent.com/bjvvuuren/predict-play/refs/heads/main/modelling/datasets/feature-engineering/rgcn_edges.csv")


In [18]:
# Identify the feature columns and row indices used by both models.
identifier_columns = {
    "node_id",
    "movieId",
    "title_id",
    "canonical_title",
    "role",
    "target"
}

feature_columns = [
    column
    for column in model_features.columns
    if column not in identifier_columns
]

X = model_features[feature_columns].to_numpy(dtype=np.float32)
y = model_features["target"].to_numpy(dtype=np.int64)
roles = model_features["role"].to_numpy()

train_index = np.flatnonzero(roles == "train")
tune_index = np.flatnonzero(roles == "tune")
calibration_index = np.flatnonzero(roles == "calibration")
catalogue_index = np.flatnonzero(roles == "catalogue")

positive_weight = float((y[train_index] == 0).sum()/(y[train_index] == 1).sum())


## Relational Graph Convolutional Network

The graph uses four metadata relations: movie-to-genre, genre-to-movie, movie-to-release-decade and release-decade-to-movie.

In [19]:
# Convert the graph node features and edges into PyTorch tensors.
node_identifier_columns = {"node_id","node_type","entity_name"}

node_feature_columns = [
    column
    for column in rgcn_node_features.columns
    if column not in node_identifier_columns
]

node_x = torch.tensor(rgcn_node_features[node_feature_columns].to_numpy(dtype=np.float32),dtype=torch.float32,device=device)

relation_names = sorted(rgcn_edges["relation"].unique().tolist())
relation_to_id = {
    relation: relation_id
    for relation_id, relation in enumerate(relation_names)
}

rgcn_edges["relation_id"] = rgcn_edges["relation"].map(relation_to_id)
edge_source = torch.tensor(rgcn_edges["source"].to_numpy(dtype=np.int64),dtype=torch.long,device=device)
edge_target = torch.tensor(rgcn_edges["target"].to_numpy(dtype=np.int64),dtype=torch.long,device=device)
edge_relation = torch.tensor(rgcn_edges["relation_id"].to_numpy(dtype=np.int64),dtype=torch.long,device=device)
movie_node_ids = torch.tensor(model_features["node_id"].to_numpy(dtype=np.int64),dtype=torch.long,device=device)

target_tensor = torch.tensor(
    y,
    dtype=torch.float32,
    device=device
)



In [20]:
# Define the R-GCN layer directly in the notebook.
class RGCNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, number_of_relations):
        super().__init__()

        self.self_linear = nn.Linear(
            input_dim,
            output_dim,
            bias=True
        )

        self.relation_weights = nn.Parameter(
            torch.empty(
                number_of_relations,
                input_dim,
                output_dim
            )
        )

        nn.init.xavier_uniform_(self.relation_weights)

    def forward(self, x, source, target, relation):
        output = self.self_linear(x)

        for relation_id in range(self.relation_weights.shape[0]):
            relation_mask = relation == relation_id

            relation_source = source[relation_mask]
            relation_target = target[relation_mask]

            messages = (
                x[relation_source]
                @ self.relation_weights[relation_id]
            )

            aggregated = torch.zeros(
                x.shape[0],
                messages.shape[1],
                device=x.device
            )

            aggregated.index_add_(
                0,
                relation_target,
                messages
            )

            degree = torch.zeros(
                x.shape[0],
                device=x.device
            )

            degree.index_add_(
                0,
                relation_target,
                torch.ones(
                    len(relation_target),
                    device=x.device
                )
            )

            output = output + (
                aggregated
                / degree.clamp(min=1).unsqueeze(1)
            )

        return output


In [21]:
# Define the two-layer R-GCN classifier directly in the notebook.
class RGCNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, number_of_relations, dropout):
        super().__init__()

        self.layer_1 = RGCNLayer(
            input_dim,
            hidden_dim,
            number_of_relations
        )

        self.layer_2 = RGCNLayer(
            hidden_dim,
            1,
            number_of_relations
        )

        self.dropout = dropout

    def forward(self, x, source, target, relation):
        x = self.layer_1(
            x,
            source,
            target,
            relation
        )

        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.layer_2(
            x,
            source,
            target,
            relation
        )

        return x.squeeze(1)


In [22]:
# Configure the R-GCN.
rgcn = RGCNClassifier(
    input_dim=node_x.shape[1],
    hidden_dim=65,
    number_of_relations=len(relation_names),
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(rgcn.parameters(),lr=0.002,weight_decay=1e-4)

loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight,dtype=torch.float32,device=device))

train_row_index = torch.tensor(train_index,dtype=torch.long,device=device)

tune_row_index = torch.tensor(tune_index,dtype=torch.long,device=device)

train_nodes = movie_node_ids[train_row_index]
tune_nodes = movie_node_ids[tune_row_index]

best_rgcn_ap = -1.0
best_rgcn_state = None
best_rgcn_epoch = 0
epochs_without_improvement = 0
max_epochs = 120
patience = 15

rgcn_history = []


In [23]:
# Train the R-GCN and use tuning average precision for early stopping.
rgcn_training_start = time.perf_counter()

for epoch in range(1, max_epochs + 1):

    rgcn.train()
    optimizer.zero_grad()

    all_node_logits = rgcn(
        node_x,
        edge_source,
        edge_target,
        edge_relation
    )

    train_logits = all_node_logits[train_nodes]
    train_targets = target_tensor[train_row_index]

    loss = loss_function(
        train_logits,
        train_targets
    )

    loss.backward()
    optimizer.step()

    rgcn.eval()

    with torch.no_grad():
        all_node_logits = rgcn(
            node_x,
            edge_source,
            edge_target,
            edge_relation
        )

        tune_probabilities = torch.sigmoid(
            all_node_logits[tune_nodes]
        ).cpu().numpy()

    tune_ap = average_precision_score(
        y[tune_index],
        tune_probabilities
    )

    rgcn_history.append({
        "epoch": epoch,
        "train_loss": float(loss.detach().cpu()),
        "tuning_ap": float(tune_ap)
    })

    if tune_ap > best_rgcn_ap:
        best_rgcn_ap = float(tune_ap)
        best_rgcn_epoch = epoch
        best_rgcn_state = copy.deepcopy(rgcn.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"loss={loss.item():.4f} | "
            f"tuning AP={tune_ap:.4f}"
        )

    if epochs_without_improvement >= patience:
        print("Early stopping at epoch", epoch)
        break

rgcn_training_seconds = time.perf_counter() - rgcn_training_start

rgcn.load_state_dict(best_rgcn_state)

torch.save(
    rgcn.state_dict(),"rgcn_weights.pt"
)

pd.DataFrame(rgcn_history).to_csv("rgcn_training_history.csv",
    index=False
)

print("Best R-GCN epoch:", best_rgcn_epoch)
print("Best tuning AP:", round(best_rgcn_ap, 4))
print("Training seconds:", round(rgcn_training_seconds, 2))


Epoch 010 | loss=0.9752 | tuning AP=0.3936
Epoch 020 | loss=0.9593 | tuning AP=0.4053
Epoch 030 | loss=0.9477 | tuning AP=0.4219
Epoch 040 | loss=0.9279 | tuning AP=0.4324
Epoch 050 | loss=0.9250 | tuning AP=0.4349
Epoch 060 | loss=0.9194 | tuning AP=0.4388
Epoch 070 | loss=0.9104 | tuning AP=0.4395
Epoch 080 | loss=0.9110 | tuning AP=0.4438
Epoch 090 | loss=0.9093 | tuning AP=0.4457
Epoch 100 | loss=0.9057 | tuning AP=0.4492
Epoch 110 | loss=0.9032 | tuning AP=0.4518
Epoch 120 | loss=0.9000 | tuning AP=0.4528
Best R-GCN epoch: 119
Best tuning AP: 0.4528
Training seconds: 43.78


In [24]:
# Produce raw R-GCN margins for every movie title.
rgcn.eval()

with torch.no_grad():
    all_node_logits = rgcn(
        node_x,
        edge_source,
        edge_target,
        edge_relation
    )

rgcn_margins = (
    all_node_logits[movie_node_ids]
    .detach()
    .cpu()
    .numpy()
)


## 4. Calibrate probability outputs

This is probability post-processing, not a third competing predictive model.

In [25]:
# Fit one sigmoid calibrator per model using only the calibration subset.
rgcn_calibrator = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=1000
)

rgcn_calibrator.fit(
    rgcn_margins[calibration_index].reshape(-1, 1),
    y[calibration_index]
)



rgcn_probabilities = rgcn_calibrator.predict_proba(
    rgcn_margins.reshape(-1, 1)
)[:, 1]


joblib.dump(
    rgcn_calibrator,"rgcn_calibrator.joblib"
)


['rgcn_calibrator.joblib']

## Compute tuning performance and score the acquisition catalogue

In [30]:
# Compare the two models on the same untouched tuning titles.
rgcn_tune_probability = rgcn_probabilities[tune_index]

model_summary = pd.DataFrame({
    "model":"R-GCN",
    "tuning_average_precision": [
        average_precision_score(
            y[tune_index],
            rgcn_tune_probability
        ),
    ],
    "training_seconds": [
        rgcn_training_seconds,
    ]
})

model_summary.to_csv("rgcn_model_summary.csv",
    index=False
)

display(model_summary)


,model,tuning_average_precision,training_seconds
0,R-GCN,0.452778,43.784582


In [27]:
tuning_prevalence = y[tune_index].mean()

print("Tuning positive rate:", round(tuning_prevalence, 4))
print("R-GCN AP:", round(best_rgcn_ap, 4))

print(
    "R-GCN lift over baseline:",
    round(best_rgcn_ap / tuning_prevalence, 2)
)

Tuning positive rate: 0.2353
R-GCN AP: 0.4528
R-GCN lift over baseline: 1.92


In [32]:
# Export acquisition-catalogue probabilities without attaching any catalogue outcome.
predictions = model_features.loc[
    catalogue_index,
    [
        "movieId",
        "title_id",
        "canonical_title"
    ]
].copy()

predictions["rgcn_p_meets_engagement_threshold"] = (
    rgcn_probabilities[catalogue_index]
)

predictions["prediction_basis"] = (
    "synthetic_tmdb_popularity_benchmarked_engagement"
)

predictions.to_csv("rgcn_catalogue_predictions.csv",
    index=False
)

display(
    predictions.sort_values(
        "rgcn_p_meets_engagement_threshold",
        ascending=False
    ).head(10)
)


,movieId,title_id,canonical_title,rgcn_p_meets_engagement_threshold,prediction_basis
6822,6934,STAD-MOV-006934,"Matrix Revolutions, The",0.923391,synthetic_tmdb_popularity_benchmarked_engagement
6260,6365,STAD-MOV-006365,"Matrix Reloaded, The",0.923391,synthetic_tmdb_popularity_benchmarked_engagement
11826,52722,STAD-MOV-052722,Spider-Man 3,0.921972,synthetic_tmdb_popularity_benchmarked_engagement
5783,5882,STAD-MOV-005882,Treasure Planet,0.907579,synthetic_tmdb_popularity_benchmarked_engagement
589,595,STAD-MOV-000595,Beauty and the Beast,0.903714,synthetic_tmdb_popularity_benchmarked_engagement
9582,30793,STAD-MOV-030793,Charlie and the Chocolate Factory,0.896341,synthetic_tmdb_popularity_benchmarked_engagement
12460,58105,STAD-MOV-058105,"Spiderwick Chronicles, The",0.888433,synthetic_tmdb_popularity_benchmarked_engagement
13095,62999,STAD-MOV-062999,Madagascar: Escape 2 Africa,0.884614,synthetic_tmdb_popularity_benchmarked_engagement
4922,5018,STAD-MOV-005018,Motorama,0.883404,synthetic_tmdb_popularity_benchmarked_engagement
12856,60684,STAD-MOV-060684,Watchmen,0.877289,synthetic_tmdb_popularity_benchmarked_engagement


In [33]:
# Export tuning predictions for later Part C metric calculations.
tuning_predictions = model_features.loc[
    tune_index,
    [
        "movieId",
        "title_id",
        "canonical_title",
        "target"
    ]
].copy()

tuning_predictions["rgcn_probability"] = (
    rgcn_probabilities[tune_index]
)



tuning_predictions.to_csv("rgcn_tuning_predictions.csv",
    index=False
)
